# Fit a simulated karyotype fitness landscape

## Goal

Simulate counts at two time points, fit a single landscape, and compare its
relative net growth rates with the simulation truth.

## Setup

Install `karstpy` in the notebook's Python environment. This example generates
its own data with fixed seeds.

In [1]:
import numpy as np
import pandas as pd
from karstpy import (
    check_sparse_joint_counts, fit_karst, predict_landscape_fitness,
    simulate_grf_landscape, simulate_sparse_counts,
)
from karstpy.karyotypes import parse_karyotypes
from karstpy.sparse_joint import self_probability

missegregation_rate = 5e-5
interval = 1
landscape = simulate_grf_landscape(
    n_chr=4, n_centroids=12, lambda_=0.8, min_cn=1, max_cn=4, seed=10,
)
predict_landscape_fitness(landscape, ["2.2.2.2"])

array([1.])

## Simulate two time points

Each column contains the counts at one time point. The count diagnostics
report how many cells share a karyotype at each time point.

In [2]:
simulation = simulate_sparse_counts(
    landscape, beta=missegregation_rate, dt=interval, n0=150, n1=150,
    seed=20, initial_population=500,
)
counts = simulation["counts"]
check_sparse_joint_counts(counts).rename(columns={"sample": "time point"}).round(3)

,time point,cells,karyotypes,repeated_fraction,nearest_distance,nullisomy_fraction,shared
0,t0,150,9,1.0,0.0,0.0,9
1,t1,150,9,1.0,0.0,0.0,9


## Fit the landscape

Supply the baseline birth rate of one per time unit and the known
missegregation rate. Relative fitness is measured from the count-weighted
mean net growth rate of the major karyotypes.

In [3]:
fit = fit_karst(
    counts, dt=interval, beta=missegregation_rate,
    sparse_joint_control={"min_cn": 1, "max_cn": 4},
)
summary = fit.summary
summary[["karyotype", "major", "shell", "relative_fitness", "relative_sd"]].head().round(5)

,karyotype,major,shell,relative_fitness,relative_sd
0,2.2.2.2,True,0.0,-0.00040,0.01164
1,1.2.2.2,True,0.0,0.00064,0.01921
2,3.2.2.2,True,0.0,0.00074,0.01921
3,2.1.2.2,True,0.0,-0.00044,0.01928
4,2.3.2.2,True,0.0,-0.00088,0.01839


## Compare with the simulation truth

The oracle returns birth rates. Multiply by `2 P_ii - 1`, where `P_ii`
is the probability that a daughter retains the parental karyotype, to get
the net growth rates estimated by the model. Subtract the same kind of
count-weighted reference from the true and fitted rates.

In [4]:
cells = summary["count_t0"] + summary["count_t1"]
is_major = summary["major"].to_numpy()
retention = self_probability(parse_karyotypes(summary["karyotype"]), missegregation_rate)
truth_net = predict_landscape_fitness(landscape, summary["karyotype"]) * (2 * retention - 1)
truth_relative = truth_net - np.average(truth_net[is_major], weights=cells[is_major])
groups = np.where(is_major, "Major", np.where(cells > 0, "Observed neighbor", "Unobserved neighbor"))
comparison = pd.DataFrame({
    "Group": groups,
    "Fitted relative growth": summary["relative_fitness"],
    "True relative growth": truth_relative,
    "Calibrated SD": summary["relative_sd"],
})
assert fit.diagnostics["finite_uncertainty"]
comparison.groupby("Group").median().round(5)

,Fitted relative growth,True relative growth,Calibrated SD
Group,,,
Major,0.00015,0.00962,0.01921
Unobserved neighbor,-0.03543,-0.04861,0.47682


## Predict in and outside the domain

These predictions share the fitted reference. The `in_domain` column
indicates whether each karyotype belongs to the fitted domain.

In [5]:
fit.predict(["2.2.2.2", "3.3.3.3"])[
    ["karyotype", "relative_fitness", "relative_sd", "in_domain"]
].round(5)

,karyotype,relative_fitness,relative_sd,in_domain
0,2.2.2.2,-0.00040,0.01164,True
1,3.3.3.3,-0.08861,1.14529,False
